# Mapping the inputs with the ids

In [4]:
# necsaire imports
import numpy as np
#from positional_enoding import generate_postionnal_encoding_matrix
tokenised_texte = ["the" , "cat" , "is" , "so" ]
d_model = 5
np.random.default_rng(42)
import numpy as np
def Mapping_inputs_tokens_with_ids(input):
    toknised_id_text = np.zeros(len(tokenised_texte))
    for i in range(0,len(tokenised_texte)):
        id = np.random.randint(low=1,high=100)
        toknised_id_text[i] = id
    return toknised_id_text

id_table = Mapping_inputs_tokens_with_ids(tokenised_texte)
print(id_table)


[43. 55. 49. 61.]


# Defining the Masked Rule

In [5]:

def create_masked_matrix():
    M = np.zeros((d_model ,d_model))
    for i in range(0,d_model):
        for j in range(0,d_model):
            if i >= j:
                M[i][j] = 0

            else:
                M[i][j] = float('-inf')

    return M

M = create_masked_matrix()
print(M)

[[  0. -inf -inf -inf -inf]
 [  0.   0. -inf -inf -inf]
 [  0.   0.   0. -inf -inf]
 [  0.   0.   0.   0. -inf]
 [  0.   0.   0.   0.   0.]]


# implementing the shfiting operation

In [9]:
def shifting_output(tokenised_texte ):
   return ["SOS"] + tokenised_texte

shifted_pos_text = shifting_output(tokenised_texte)


# calculation of Xdec

In [10]:

from positional_enoding import generate_postionnal_encoding_matrix
PE_matrix = generate_postionnal_encoding_matrix(tokenised_texte=shifted_pos_text , d_model=5)   
print(PE_matrix)
Xe = np.random.rand(d_model,d_model)
Wq_self = np.random.rand(d_model,d_model)
Wk_self = np.random.rand(d_model,d_model)
Wv_self = np.random.rand(d_model,d_model)
WO_self = np.random.rand(d_model,d_model)
X_dec = Xe + PE_matrix

[[0.0, 1.0, 0.0, 1.0, 0.0], [0.8414709848078965, 0.5403023058681398, 0.025116222909773774, 0.9996845379152098, 0.0006309573026154199], [0.9092974268256817, -0.4161468365471424, 0.050216599387465206, 0.9987383506934931, 0.0012619143540422218], [0.1411200080598672, -0.9899924966004454, 0.07528529299888895, 0.997162035307237, 0.0018928709030918876], [-0.7568024953079282, -0.6536436208636119, 0.10030648729934574, 0.9949565862919176, 0.0025238266985760983]]


# Calculating the decoder self attention

In [ ]:
from positional_enoding import layer_normalisation
test = [1,2,3]
def softmax(x):
    # Subtract np.max for numerical stability (prevents overflow)
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

def Relu(x):
    return np.maximum(0,x)


def decoder_attention(X_dec):
    
    Q_self = np.matmul(X_dec,Wq_self)
    K_self = np.matmul(X_dec,Wk_self)
    V = np.matmul(X_dec,Wv_self)
    scores = np.dot(Q_self, K_self.T) / np.sqrt(d_model)
    scores = softmax(scores)
    scores = np.dot(scores,V)
    A_masked = (scores + M)
    out = np.dot(A_masked,V)
    Y_self = np.dot(out , WO_self)
    return Y_self

Z1 = decoder_attention(X_dec) + X_dec
X_1 = layer_normalisation(Z1)



# calculating the cross attention

In [21]:
H_encoder = np.array([-1.22880511, 1.79518924, -0.78551144, 0.64881799, 
                      1.03376784, 0.39942824, -0.68147284, 1.38844063, 
                      0.42401203, -1.40880753, -0.80973162, 1.17463362, 
                      -0.30808342, -1.4134708, -0.49242468, 0.26401786])
H_encoder = H_encoder.reshape(4,4)
Wq_cross = np.random.rand(d_model,d_model)
Wk_cross = np.random.rand(d_model,d_model)
Wv_cross = np.random.rand(d_model,d_model)
WO_cross = np.random.rand(d_model,d_model)
def cross_attention(X):
    Q_cross = np.dot(X_1,Wq_cross)
    K_cross = np.dot(H_encoder,Wk_cross)
    V_cross = np.dot(H_encoder,Wv_cross)
    scores_cross_attention = np.dot(Q_cross , K_cross.T) / np.sqrt(d_model)
    A_cross = softmax(scores_cross_attention)
    Y_cross = np.dot(A_cross,V_cross)
    Y_cross = np.dot(Y_cross , WO_cross)
    Z2 = X_1 + Y_cross
    X_2 = layer_normalisation(Z2)
    W1 = np.random.randn(d_model , d_model)
    W2 = np.random.randn(d_model , d_model)
    Y_fnn = Relu(np.dot(X_2,W1)) 
    Y_fnn = np.dot(Y_fnn , W2)
    Z3 = X_2 + Y_fnn
    Z3 = layer_normalisation(Z3)
    H_dec  = layer_normalisation(Z3)
    return H_dec
    

# final projection

In [25]:
H_dec = cross_attention(X_dec)

ValueError: shapes (4,4) and (5,5) not aligned: 4 (dim 1) != 5 (dim 0)